# MedAlign — Generate DPO Preference Pairs (Kaggle GPU)
Sample multiple responses from the SFT model at varying temperatures, judge with GPT-4, build (prompt, chosen, rejected) triples → push to HF Hub.

In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate datasets huggingface_hub openai

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
u = UserSecretsClient()
os.environ['HF_TOKEN'] = u.get_secret('HF_TOKEN')
os.environ['OPENAI_API_KEY'] = u.get_secret('OPENAI_API_KEY')
from huggingface_hub import login; login(os.environ['HF_TOKEN'])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
BASE='mistralai/Mistral-7B-Instruct-v0.3'
ADAPTER='Zubairash/medalign-sft-adapters'
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)
tok=AutoTokenizer.from_pretrained(BASE)
base=AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model=PeftModel.from_pretrained(base, ADAPTER)
model.eval()

In [ ]:
from datasets import load_dataset
prompts = load_dataset('medalpaca/medical_meadow_medqa', split='train').select(range(2000))

def sample(prompt, temp):
    inp = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=256, do_sample=True, temperature=temp, top_p=0.9)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

samples = []
for ex in prompts:
    p = ex['instruction']
    samples.append({'prompt': p, 'a': sample(p, 0.3), 'b': sample(p, 0.9)})
len(samples)

In [ ]:
# Judge with GPT-4 → mark which is chosen vs rejected
from openai import OpenAI
client = OpenAI()
JUDGE_PROMPT = (
  'You are a medical expert. Given a question and two answers, choose the more accurate, '
  'safe, and complete one. Reply with only A or B.\n\nQ: {q}\n\nA: {a}\n\nB: {b}'
)
pairs = []
for s in samples:
    r = client.chat.completions.create(model='gpt-4o-mini', messages=[{'role':'user','content':JUDGE_PROMPT.format(q=s['prompt'], a=s['a'], b=s['b'])}])
    choice = r.choices[0].message.content.strip()[:1].upper()
    if choice not in ('A','B'): continue
    chosen, rejected = (s['a'], s['b']) if choice=='A' else (s['b'], s['a'])
    pairs.append({'prompt': s['prompt'], 'chosen': chosen, 'rejected': rejected})
len(pairs)

In [ ]:
from datasets import Dataset
Dataset.from_list(pairs).push_to_hub('Zubairash/medalign-dpo-pairs', private=False)